# DNAmStress

## Index
1. [Instantiate model class](#Instantiate-model-class)
2. [Define clock metadata](#Define-clock-metadata)
3. [Download clock dependencies](#Download-clock-dependencies)
5. [Load features](#Load-features)
6. [Load weights into base model](#Load-weights-into-base-model)
7. [Load reference values](#Load-reference-values)
8. [Load preprocess and postprocess objects](#Load-preprocess-and-postprocess-objects)
10. [Check all clock parameters](#Check-all-clock-parameters)
10. [Basic test](#Basic-test)
11. [Save torch model](#Save-torch-model)
12. [Clear directory](#Clear-directory)

Let's first import some packages:

In [1]:
import os
import inspect
import shutil
import json
import torch
import pandas as pd
import pyaging as pya

## Instantiate model class

In [2]:
def print_entire_class(cls):
    source = inspect.getsource(cls)
    print(source)

print_entire_class(pya.models.DNAmStress)

class DNAmStress(LinearReferenceClock):
    pass



In [3]:
model = pya.models.DNAmStress()

## Define clock metadata

In [4]:
model.metadata["clock_name"] = "dnamstress"
model.metadata["data_type"] = "DNA methylation"  # Paper: Whole-blood DNA methylation was quantified for the epigenomic cohort.
model.metadata["species"] = "Homo sapiens"  # Paper: The NIAAA cohorts comprised human participants with AUD and healthy controls.
model.metadata["year"] = 2023
model.metadata["approved_by_author"] = "⌛"
model.metadata["citation"] = "Jung, J., et al. “Additive Effects of Stress and Alcohol Exposure on Accelerated Epigenetic Aging in Alcohol Use Disorder.” Biological Psychiatry 93(4): 331–341 (2023)."
model.metadata["doi"] = "https://doi.org/10.1016/j.biopsych.2022.06.036"
model.metadata["notes"] = "Whole-blood methylation score (MS stress) derived as a 211-CpG proxy for a composite of 13 stress-related measures."
model.metadata["research_only"] = None
model.metadata["tissue"] = ["whole blood"]  # Paper: Whole-blood DNAm was quantified in all 615 epigenomic-cohort participants.
model.metadata["predicts"] = ["stress exposure"]  # Paper: MS stress is a methylation-derived score tracking stress exposure.
model.metadata["training_target"] = ["stress exposure"]  # Paper: Three factor scores from 13 stress-associated measures were added to construct the composite score of stress.
model.metadata["unit"] = ["unitless"]  # Paper: MS stress was calculated as the weighted sum of selected CpG methylation values.
model.metadata["model_type"] = "bootstrap-stabilized elastic net regression"  # Paper: An elastic net model with 1,000 bootstraps selected CpGs present in more than half of bootstraps.
model.metadata["platform"] = ["Illumina EPIC"]  # Paper: Whole-blood DNAm was quantified using Infinium MethylationEPIC BeadChip arrays.
model.metadata["population"] = "adults"  # Paper: The epigenomic cohort consisted of 615 participants: 372 AUD cases and 243 healthy controls.
model.metadata["journal"] = "Biological Psychiatry"
model.metadata["last_author"] = "Falk W. Lohoff"
model.metadata["n_features"] = 211
model.metadata["citations"] = 27
model.metadata["citations_date"] = "2026-07-05"


## Download clock dependencies

In [5]:
os.system(f"curl -sL -o DNAmStress_CpGs.rda https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data/DNAmStress_CpGs.rda")
os.system(f"curl -sL -o DNAmStress_ref.rda https://raw.githubusercontent.com/HigginsChenLab/methylCIPHER/19b12296b0d7eb7055a97d068064df635f44ce3e/data/DNAmStress_ref.rda")

0

In [6]:
%%writefile download.r

library(jsonlite)
load("DNAmStress_CpGs.rda")
load("DNAmStress_ref.rda")
write_json(DNAmStress_CpGs, "coefficients.json", digits = 10)
write_json(DNAmStress_ref$intercept, "intercept.json", digits = 10)

Writing download.r


In [7]:
os.system("Rscript download.r")

0

## Load features

In [8]:
coef_df = pd.DataFrame(json.load(open('coefficients.json')))
model.features = coef_df['CpG'].tolist()
iv = json.load(open('intercept.json'))
intercept_value = iv[0] if isinstance(iv, list) else iv

## Load weights into base model

In [9]:
weights = torch.tensor(coef_df['coefficient'].tolist()).unsqueeze(0).float()
intercept = torch.tensor([intercept_value]).float()

In [10]:
base_model = pya.models.LinearModel(input_dim=len(model.features))

base_model.linear.weight.data = weights.float()
base_model.linear.bias.data = intercept.float()

model.base_model = base_model

## Load reference values

In [11]:
model.reference_values = None

## Load preprocess and postprocess objects

In [12]:
model.preprocess_name = None
model.preprocess_dependencies = None

In [13]:
model.postprocess_name = None
model.postprocess_dependencies = None

## Check all clock parameters

In [14]:
pya.utils.print_model_details(model)


%==================================== Model Details ====================================%
Model Attributes:

training: True
metadata: {'approved_by_author': '⌛',
 'citation': 'Jung, Sun Jae, et al. "An epigenetic biomarker of stress and its '
             'association with cardiovascular disease." Biological Psychiatry '
             '93.4 (2023): 331-341.',
 'clock_name': 'dnamstress',
 'data_type': 'methylation',
 'doi': 'https://doi.org/10.1016/j.biopsych.2022.06.036',
 'notes': None,
 'research_only': None,
 'species': 'Homo sapiens',
 'version': None,
 'year': 2023}
reference_values: None
preprocess_name: None
preprocess_dependencies: None
postprocess_name: None
postprocess_dependencies: None
features: ['cg00188055', 'cg00418087', 'cg00452133', 'cg00599628', 'cg00684116', 'cg00728602', 'cg01047613', 'cg01085421', 'cg01245530', 'cg01607512', 'cg01631065', 'cg02107357', 'cg02386470', 'cg02606284', 'cg02606535', 'cg02607810', 'cg02652698', 'cg02873991', 'cg02978227', 'cg03104567', '

## Basic test

In [15]:
torch.manual_seed(42)
input = torch.randn(10, len(model.features), dtype=float)
model.eval()
model.to(float)
pred = model(input)
pred

tensor([[-33.9411],
        [-62.7000],
        [ 93.3530],
        [ 14.1386],
        [-18.2623],
        [  4.9429],
        [ 94.5241],
        [ 63.0782],
        [  4.6987],
        [ 66.1088]], dtype=torch.float64, grad_fn=<AddmmBackward0>)

## Save torch model

In [16]:
torch.save(model, f"../weights/{model.metadata['clock_name']}.pt")

## Clear directory
<a id="10"></a>

In [17]:
# Function to remove a folder and all its contents
def remove_folder(path):
    try:
        shutil.rmtree(path)
        print(f"Deleted folder: {path}")
    except Exception as e:
        print(f"Error deleting folder {path}: {e}")

# Get a list of all files and folders in the current directory
all_items = os.listdir('.')

# Loop through the items
for item in all_items:
    # Check if it's a file and does not end with .ipynb
    if os.path.isfile(item) and not item.endswith('.ipynb'):
        os.remove(item)
        print(f"Deleted file: {item}")
    # Check if it's a folder
    elif os.path.isdir(item):
        remove_folder(item)

Deleted file: intercept.json
Deleted file: DNAmStress_CpGs.rda
Deleted file: download.r
Deleted file: coefficients.json
Deleted file: DNAmStress_ref.rda
